
# Y-Maze DeepLabCut Output Quality Control Notebook

This notebook provides a workflow for quality control of DeepLabCut output files from Y-maze behavioral experiments. It automates the following steps:

- **Video File Validation:** Scans a specified folder for MP4 video files, checks their frame counts, and flags videos exceeding a defined frame threshold.
- **DeepLabCut Output Inspection:** Loads all DeepLabCut `.h5` files in the folder, counts the number of rows (frames) in each, and summarizes the results.
- **Consistency Checks:** Compares frame counts across all `.h5` files to ensure data integrity and consistency.
- **Reporting:** Saves summary tables (frame counts and row counts) as CSV files for record-keeping and further analysis.

This notebook is intended to help researchers quickly identify problematic files and verify that all DeepLabCut outputs are complete and consistent before downstream analysis.

In [12]:
import cv2
import os
import pandas as pd
from pathlib import Path
import tkinter as tk
from tkinter import filedialog
# This script creates a dictionary mapping video filenames to their full paths

# Folder containing video files
root = tk.Tk()
root.withdraw()
folder_path = filedialog.askdirectory(title="Select folder with video files")


# Ensure the folder path is correct
if not os.path.exists(folder_path):
    print(f"Folder does not exist: {folder_path}")
    exit(1)


print("Found video files:", folder_path)



Found video files: D:/KarenDuff/Ymaze


## Use the code below to check number of frames in .mp4 files 

In [10]:
# Max frame threshold
frame_threshold = 18300

# Get list of MP4 files
video_files = [f for f in os.listdir(folder_path) if f.endswith('.mp4')]

if not video_files:
    print("No MP4 files found in the specified folder.")
    exit(1)

print("Found video files:", video_files)


# Collect data
data = []

for file in video_files:
    file_path = os.path.join(folder_path, file)
    cap = cv2.VideoCapture(file_path)
    
    if not cap.isOpened():
        print(f"Error opening video: {file}")
        continue

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    status = "OK" if total_frames < frame_threshold else "Too many frames"
    data.append({"File": file, "Frame Count": total_frames, "Status": status})
    cap.release()

# Create DataFrame
df = pd.DataFrame(data)

# Display results
print(df)

# Save to CSV
output_csv_path = os.path.join(folder_path, "video_frame_summary.csv")
df.to_csv(output_csv_path, index=False)
print(f"\n📄 Results saved to: {output_csv_path}")

# Check min and max frame counts
if data:
    frame_counts = [d['Frame Count'] for d in data]
    min_frames = min(frame_counts)
    max_frames = max(frame_counts)
    print(f"\n✅ Minimum frame count: {min_frames}")
    print(f"✅ Maximum frame count: {max_frames}")
else:
    print("\n⚠️ No valid MP4 files were processed.")

Found video files: ['ELGH-1945_Ymaze_20240222_143607191_downsample.mp4', 'ELGH-1946_Ymaze_20240222_144558804_downsample.mp4', 'ELGH-2391_Ymaze_20240222_162742885_downsample.mp4', 'ELGH-2394_Ymaze_20240222_163723613_downsample.mp4', 'ELGH-2410_Ymaze_20240307_151332154downsampled.mp4', 'ELGH-2411_Ymaze_20240307_150417566downsampled.mp4', 'ELGH-2420_Ymaze_20240222_164717155_downsample.mp4', 'ELGH-2515_Ymaze_20240307_153211062downsampled.mp4', 'ELGH-2516_Ymaze_20240307_154121850downsampled.mp4', 'ELGH-2517_Ymaze_20240307_155050711downsampled.mp4', 'ELGH-2518_Ymaze_20240307_155957101downsampled.mp4', 'ELGH-2526_Ymaze_20240307_152303744downsampled.mp4', 'ELGH-2530_Ymaze_20240222_165657690_downsample.mp4', 'ELGH-2531_Ymaze_20240222_170634011_downsample.mp4', 'ELGH-2559_Ymaze_20240222_145637806downsampled.mp4', 'ELGH-2559_Ymaze_20240222_145637806_Graded2downsampled.mp4', 'ELGH-2559_Ymaze_20240222_145637806_Gradeddownsampled.mp4', 'ELGH-2561_Ymaze_20240222_150745610_downsample.mp4', 'ELGH-2574_

# Use the script bellow to check h5 files 

In [14]:
import pandas as pd
import os

# Folder containing DeepLabCut .h5 files
folder_path = filedialog.askdirectory(title="Select folder with DeepLabCut .h5 files")
# Ensure the folder path is correct
if not os.path.exists(folder_path):
    print(f"Folder does not exist: {folder_path}")
    exit(1)
print("Found DeepLabCut .h5 files:", folder_path)


# Get all .h5 files
h5_files = [f for f in os.listdir(folder_path) if f.endswith('.h5')]

# Collect results
row_counts = []

for file in h5_files:
    file_path = os.path.join(folder_path, file)
    try:
        df = pd.read_hdf(file_path)
        num_rows = len(df)
        print(f"{file}: {num_rows} rows")
        row_counts.append((file, num_rows))
    except Exception as e:
        print(f"❌ Failed to read {file}: {e}")

# Check min and max if we got any valid files
if row_counts:
    row_values = [r[1] for r in row_counts]
    min_rows = min(row_values)
    max_rows = max(row_values)
    print(f"\n✅ Minimum rows: {min_rows}")
    print(f"✅ Maximum rows: {max_rows}")
else:
    print("\n⚠️ No valid .h5 files were processed.")


Found DeepLabCut .h5 files: D:/KarenDuff/Ymaze

⚠️ No valid .h5 files were processed.


In [ ]:
df_summary = pd.DataFrame(row_counts, columns=["File", "Rows"])
df_summary.to_csv(os.path.join(folder_path, "deeplabcut_row_counts.csv"), index=False)
